# Human check
- A judge labels test clips from the 9 frames only (no WAD reminder, no module output, no Qwen reply).
- For each clip: the main hazard, its direction, the action, and the move direction if the action is "move".

**Instructions for the judge**
- Imagine guiding a blind person who is walking with a cane. Looking at the frames (frame 8 = now), answer:
    - what is the single most important thing they should be told about in the next few seconds, where is it,
    - and what should they do? If nothing needs a warning, choose 0 (nothing in the way) and "continue".

## 1. Settings
- `JUDGE = 1` or `2`. Each judge has their own labels file.
- `FRAMES_DIR`: the `frames` folder from the unzipped `wad_test.zip`.

In [ ]:
import time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import clear_output

JUDGE = 2
N_THIS_SESSION = 100 # clips to label each session
SEED = 7

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FRAMES_DIR = Path("/Users/amynguyen/Documents/MQ/(26_2) FOSE7902 - Advanced STEM Research Placement/WAD/wad_dataset_v2/src_data")
OUT = REPO_ROOT / "results" / "test" / "human"
OUT.mkdir(parents=True, exist_ok=True)
LABELS = REPO_ROOT / "results" / "outputs" / "test" / "human" / f"blind_labels_judge{JUDGE}.csv"

## 2. The sample
- 350 test clips with the same danger level × response type mix as the test set.
- 50 shared clips (both judges) + 150 per judge; each judge sees their 200 in a random order.

In [ ]:
test_ids = pd.read_csv(REPO_ROOT / "data" / "subsets" / "wad_test.csv")
test_ids["stratum"] = test_ids.danger_level + "|" + test_ids.response_type

def strat_sample(df, n, seed):
    """n clips with the same danger x response mix as df."""
    k = (df.stratum.value_counts(normalize=True) * n).round().astype(int)
    k.iloc[0] += n - k.sum()
    return pd.concat([df[df.stratum == s].sample(m, random_state=seed) for s, m in k.items() if m > 0])

pool = strat_sample(test_ids, 350, SEED) # all clips in the check
shared = strat_sample(pool, 50, SEED) # labelled by both judges
rest = pool.drop(shared.index)
own1 = strat_sample(rest, 150, SEED) # judge 1 only
own2 = rest.drop(own1.index) # judge 2 only

SAMPLE = list(pd.concat([shared, own1 if JUDGE == 1 else own2]).sample(frac=1, random_state=SEED + JUDGE).frame_path)
assert all((FRAMES_DIR / c / f"{k}.jpg").exists() for c in SAMPLE for k in range(9))

## 3. Answer options

In [ ]:
HAZARDS = ["pedestrian", "vehicle", "two_wheeler", "pole", "tree_plant", "sign", "fence_railing", "barrier_cone",
           "furniture_stall", "stairs_step", "curb_edge", "water", "hole_uneven", "wall_door", "animal",
           "construction", "street_object", "junction", "narrow_crowded", "other"]
ACTIONS = {"c": "continue", "s": "slow", "m": "move", "x": "stop"}
CLOCKS = {str(h) for h in range(1, 13)}
MENU = "  ".join(f"{i}={h}" for i, h in enumerate(HAZARDS, 1))

def ask(question, valid): # ask till the answer is valid
    while True:
        a = input(question).strip().lower()
        if a in valid:
            return a

def show(clip, title):
    fig, axs = plt.subplots(3, 3, figsize=(14, 8))
    for k, ax in enumerate(axs.flat):
        im = Image.open(FRAMES_DIR / clip / f"{k}.jpg")
        im.thumbnail((640, 640))                      
        ax.imshow(im)
        ax.set_title(f"frame {k}" + ("  (now)" if k == 8 else ""), fontsize=9)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 4. Label (run again to continue)
- Hazard: type its number, or 0 if nothing is in the way. Direction: 12 = straight ahead, 3 = right, 9 = left.
- Each answer is saved immediately.

In [5]:
done = set(pd.read_csv(LABELS).frame_path) if LABELS.exists() else set()
todo = [c for c in SAMPLE if c not in done][:N_THIS_SESSION]

for i, clip in enumerate(todo, 1):
    clear_output(wait=True)
    show(clip, f"judge {JUDGE} | clip {len(done) + i} of {len(SAMPLE)}")
    t0 = time.time()
    n = f"[clip {len(done) + i}/{len(SAMPLE)}]"
    h = int(ask(f"{n} 1/3 Main hazard (0 = nothing in the way)\n{MENU}\n> ", {str(n_) for n_ in range(len(HAZARDS) + 1)}))
    clock = ask(f"{n} 2/3 Its direction (clock 1-12) > ", CLOCKS) if h else ""
    action = ACTIONS[ask(f"{n} 3/3 Action: c = continue, s = slow down, m = move aside, x = stop > ", set(ACTIONS))]
    move_clock = ask(f"{n} Move towards which clock (1-12)? > ", CLOCKS) if action == "move" else ""
    row = dict(frame_path=clip, judge=JUDGE, hazard=HAZARDS[h - 1] if h else "none", clock=clock,
               action=action, move_clock=move_clock, seconds=round(time.time() - t0, 1))
    pd.DataFrame([row]).to_csv(LABELS, mode="a", header=not LABELS.exists(), index=False)

clear_output()
print(f"judge {JUDGE}: {len(done) + len(todo)} of {len(SAMPLE)} clips labelled")

judge 2: 200 of 200 clips labelled
